In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os

# Set device to GPU if available (Crucial for your ROCm/CUDA background!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import os
import urllib.request

# Create a data directory outside the repo
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)

# Using the NERSC open directory which allows direct Python downloads
base_url = "https://portal.nersc.gov/cfs/m4392/G25/"

files_to_download = [
    "Dataset_Specific_Unlabelled.h5",
    "Dataset_Specific_labelled_full_only_for_2i.h5"
]

for filename in files_to_download:
    url = base_url + filename
    filepath = os.path.join(data_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} from NERSC... (This is a large file, it may take a few minutes)")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Successfully downloaded {filename}!")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
    else:
        print(f"{filename} already exists. Skipping download.")

print("All data ready!")

In [ ]:
class CMSParticleDataset(Dataset):
    def __init__(self, file_path, is_labelled=True):
        """
        Custom PyTorch Dataset for CMS .h5 particle collision files.
        """
        self.file_path = file_path
        self.is_labelled = is_labelled
        
        # Open HDF5 file
        self.h5_file = h5py.File(file_path, 'r')
        
        # Extract images (Assuming the key is 'X_jet' based on typical CMS datasets)
        # Note: We keep them on disk and load iteratively to save RAM
        self.images = self.h5_file['X_jet'] 
        
        if self.is_labelled:
            self.labels = self.h5_file['y']       # Classification targets
            self.masses = self.h5_file['am']      # Regression targets

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image array and convert to float32 tensor
        # Typically CMS images are (H, W, Channels). PyTorch needs (Channels, H, W)
        img_np = np.array(self.images[idx], dtype=np.float32)
        
        # Rearrange dimensions if necessary (Uncomment if needed after inspecting data shape)
        if img_np.shape[-1] <= 4: # Assuming max 4 channels (e.g., Track pT, DZ, D0, ECAL)
            img_np = np.transpose(img_np, (2, 0, 1))
            
        img = torch.tensor(img_np)
        
        if self.is_labelled:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            mass = torch.tensor(self.masses[idx], dtype=torch.float32)
            return img, label, mass
        else:
            return img

# --- Let's test it and print the shapes ---
unlabelled_path = '../data/Dataset_Specific_Unlabelled.h5'
if os.path.exists(unlabelled_path):
    print("Keys in unlabelled file:", list(h5py.File(unlabelled_path, 'r').keys()))
    dummy_dataset = CMSParticleDataset(unlabelled_path, is_labelled=False)
    print(f"Unlabelled dataset size: {len(dummy_dataset)}")
    print(f"Sample image shape: {dummy_dataset[0].shape}")
else:
    print("Dataset not found. Please run the download cell.")